# Cell 1: Mount Drive and define manifest/model directories

In [1]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
# Ensure this points to the manifest file generated in Phase 1
MANIFEST_PATH = Path("/content/drive/MyDrive/Lung_Nodule_Project/processed_patches/manifest.csv")
MODEL_DIR = Path("/content/drive/MyDrive/Lung_Nodule_Project/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Manifest Path: {MANIFEST_PATH}")
print(f"Models Save Directory: {MODEL_DIR}")


ModuleNotFoundError: No module named 'google'

# Cell 2: Patch NumPy variables and implement dynamic radiomic extraction

In [ ]:
import numpy as np
import scipy.ndimage as ndimage
from typing import Tuple, Dict
# Fix pylidc compatibility with modern NumPy in case pylidc is imported later
# Pass cleanly without modifying np attributes,
# or use built-in types directly in function signatures/casts.
def generate_approximate_nodule_mask(patch_np: np.ndarray, center_zyx: Tuple[int,int,int] = (32,32,32), intensity_threshold: float = 0.35, max_radius: float = 20.0) -> np.ndarray:
    z, y, x = np.ogrid[:patch_np.shape[0], :patch_np.shape[1], :patch_np.shape[2]]
    dist_sq = (z - center_zyx[0])**2 + (y - center_zyx[1])**2 + (x - center_zyx[2])**2
    distance_mask = dist_sq <= max_radius**2
    threshold_mask = patch_np > intensity_threshold
    combined_mask = threshold_mask & distance_mask

    labeled_mask, num_features = ndimage.label(combined_mask)
    if num_features == 0:
        return distance_mask & (dist_sq <= 5.0**2)

    center_label = labeled_mask[center_zyx[0], center_zyx[1], center_zyx[2]]
    if center_label > 0:
        final_mask = labeled_mask == center_label
    else:
        component_centers = ndimage.center_of_mass(combined_mask, labeled_mask, range(1, num_features + 1))
        if num_features == 1:
            component_centers = [component_centers]

        min_dist = float('inf')
        closest_label = 1
        for i, c_com in enumerate(component_centers):
            if c_com is None or np.isnan(c_com).any():
                continue
            dist = np.sum((np.array(c_com) - np.array(center_zyx))**2)
            if dist < min_dist:
                min_dist = dist
                closest_label = i + 1
        final_mask = labeled_mask == closest_label
    return final_mask.astype(bool)
def compute_first_order_features(patch_np: np.ndarray, mask_np: np.ndarray) -> Dict[str, float]:
    nodule_voxels = patch_np[mask_np]
    if len(nodule_voxels) == 0:
        return {f"fo_{k}": 0.0 for k in ["mean","std","skew","kurt","entropy","energy","range","uniformity"]}

    mean, std = float(np.mean(nodule_voxels)), float(np.std(nodule_voxels))
    skew = float(np.mean(((nodule_voxels - mean)/std)**3)) if std > 0 else 0.0
    kurt = float(np.mean(((nodule_voxels - mean)/std)**4)) - 3.0 if std > 0 else 0.0
    r_min, r_max = float(np.min(nodule_voxels)), float(np.max(nodule_voxels))
    val_range = r_max - r_min

    hist, _ = np.histogram(nodule_voxels, bins=32, range=(0.0, 1.0), density=True)
    hist = hist / (np.sum(hist) + 1e-8)
    entropy = -float(np.sum(hist * np.log2(hist + 1e-8)))
    uniformity = float(np.sum(hist ** 2))
    energy = float(np.sum(nodule_voxels ** 2))

    return {
        "fo_mean": mean, "fo_std": std, "fo_skew": skew, "fo_kurt": kurt,
        "fo_entropy": entropy, "fo_energy": energy, "fo_range": val_range, "fo_uniformity": uniformity
    }
def compute_shape_features(mask_np: np.ndarray) -> Dict[str, float]:
    voxel_count = float(np.sum(mask_np))
    if voxel_count == 0:
        return {"shape_volume": 0.0, "shape_surface_area": 0.0, "shape_sphericity": 0.0, "shape_compactness": 0.0}
    volume = voxel_count
    dilated = ndimage.binary_dilation(mask_np)
    boundary_voxels = np.sum(dilated ^ mask_np)
    surface_area = float(boundary_voxels)
    sphericity = (np.pi**(1/3) * (6 * volume)**(2/3)) / surface_area if surface_area > 0 else 0.0
    compactness = volume / (surface_area ** 1.5) if surface_area > 0 else 0.0
    return {
        "shape_volume": volume, "shape_surface_area": surface_area,
        "shape_sphericity": min(1.0, float(sphericity)), "shape_compactness": float(compactness)
    }
def compute_glcm_texture_features(patch_np: np.ndarray, mask_np: np.ndarray) -> Dict[str, float]:
    axial_sums = np.sum(mask_np, axis=(1, 2))
    center_z = int(np.argmax(axial_sums))
    slice_img, slice_mask = patch_np[center_z, :, :], mask_np[center_z, :, :]
    if np.sum(slice_mask) < 4:
        return {f"glcm_{k}": 0.0 for k in ["contrast","correlation","homogeneity","energy"]}

    gray_levels = 8
    discretized = np.clip(np.round(slice_img * (gray_levels - 1)), 0, gray_levels - 1).astype(int)
    glcm = np.zeros((gray_levels, gray_levels), dtype=float)
    h, w = slice_img.shape
    count = 0
    for y in range(h):
        for x in range(w - 1):
            if slice_mask[y, x] and slice_mask[y, x + 1]:
                glcm[discretized[y, x], discretized[y, x + 1]] += 1.0
                count += 1
    if count == 0:
        return {f"glcm_{k}": 0.0 for k in ["contrast","correlation","homogeneity","energy"]}

    glcm /= count
    i_indices, j_indices = np.ogrid[:gray_levels, :gray_levels]
    mean_i = np.sum(i_indices * np.sum(glcm, axis=1))
    mean_j = np.sum(j_indices * np.sum(glcm, axis=0))
    std_i = np.sqrt(np.sum((i_indices - mean_i)**2 * np.sum(glcm, axis=1)) + 1e-8)
    std_j = np.sqrt(np.sum((j_indices - mean_j)**2 * np.sum(glcm, axis=0)) + 1e-8)
    contrast, homogeneity, energy, correlation = 0.0, 0.0, 0.0, 0.0
    for i in range(gray_levels):
        for j in range(gray_levels):
            prob = glcm[i, j]
            contrast += prob * (i - j)**2
            homogeneity += prob / (1.0 + abs(i - j))
            energy += prob**2
            correlation += prob * (i - mean_i) * (j - mean_j) / (std_i * std_j)

    return {"glcm_contrast": float(contrast), "glcm_correlation": float(correlation), "glcm_homogeneity": float(homogeneity), "glcm_energy": float(energy)}
def extract_all_radiomics(patch_np: np.ndarray) -> np.ndarray:
    mask_np = generate_approximate_nodule_mask(patch_np)
    fo = compute_first_order_features(patch_np, mask_np)
    shape = compute_shape_features(mask_np)
    texture = compute_glcm_texture_features(patch_np, mask_np)

    features = [
        fo["fo_mean"], fo["fo_std"], fo["fo_skew"], fo["fo_kurt"], fo["fo_entropy"], fo["fo_energy"], fo["fo_range"], fo["fo_uniformity"],
        shape["shape_volume"], shape["shape_surface_area"], shape["shape_sphericity"], shape["shape_compactness"],
        texture["glcm_contrast"], texture["glcm_correlation"], texture["glcm_homogeneity"], texture["glcm_energy"]
    ]
    return np.array(features, dtype=np.float32)

# Cell 3: Define 3D CNN, Tabular MLP, Gated Fusion and predicting heads

In [ ]:
import torch
import torch.nn as nn
class ConvBlock3D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout_prob: float = 0.1):
        super().__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm3d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout3d(dropout_prob)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.relu(self.bn(self.conv(x))))
class Spatial3DCNNEncoder(nn.Module):
    def __init__(self, in_channels: int = 1, latent_dim: int = 128):
        super().__init__()
        self.block1 = ConvBlock3D(in_channels, 32)
        self.pool1 = nn.MaxPool3d(2, 2)
        self.block2 = ConvBlock3D(32, 64)
        self.pool2 = nn.MaxPool3d(2, 2)
        self.block3 = ConvBlock3D(64, 128)
        self.pool3 = nn.MaxPool3d(2, 2)
        self.block4 = ConvBlock3D(128, 256)
        self.pool4 = nn.MaxPool3d(2, 2)
        self.gap = nn.AdaptiveAvgPool3d((1,1,1))
        self.flatten = nn.Flatten()
        self.fc_proj = nn.Sequential(nn.Linear(256, latent_dim), nn.ReLU(inplace=True), nn.Dropout(0.2))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.pool1(self.block1(x))
        out = self.pool2(self.block2(out))
        out = self.pool3(self.block3(out))
        out = self.pool4(self.block4(out))
        return self.fc_proj(self.flatten(self.gap(out)))
class RadiomicsFeatureEncoder(nn.Module):
    def __init__(self, input_dim: int = 21, latent_dim: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(128, latent_dim), nn.BatchNorm1d(latent_dim), nn.ReLU(inplace=True), nn.Dropout(0.2)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)
class GatedMultimodalFusion(nn.Module):
    def __init__(self, cnn_dim: int = 128, rad_dim: int = 64, fused_dim: int = 128):
        super().__init__()
        self.proj_rad = nn.Linear(rad_dim, cnn_dim)
        self.gate_cnn = nn.Sequential(nn.Linear(cnn_dim, cnn_dim), nn.Sigmoid())
        self.gate_rad = nn.Sequential(nn.Linear(cnn_dim, cnn_dim), nn.Sigmoid())
        self.out_proj = nn.Sequential(nn.Linear(cnn_dim, fused_dim), nn.ReLU(inplace=True), nn.Dropout(0.3))
    def forward(self, x_cnn: torch.Tensor, x_rad: torch.Tensor) -> torch.Tensor:
        x_rad_proj = self.proj_rad(x_rad)
        g_cnn = self.gate_cnn(x_cnn)
        g_rad = self.gate_rad(x_rad_proj)
        return self.out_proj(g_cnn * x_cnn + g_rad * x_rad_proj)
class HybridMalignancyNet(nn.Module):
    def __init__(self, cnn_in_channels: int = 1, rad_in_features: int = 21, cnn_latent_dim: int = 128, rad_latent_dim: int = 64, fused_dim: int = 128):
        super().__init__()
        self.cnn_encoder = Spatial3DCNNEncoder(cnn_in_channels, cnn_latent_dim)
        self.rad_encoder = RadiomicsFeatureEncoder(rad_in_features, rad_latent_dim)
        self.fusion = GatedMultimodalFusion(cnn_latent_dim, rad_latent_dim, fused_dim)
        self.clf_head = nn.Linear(fused_dim, 2)
        self.reg_head = nn.Sequential(nn.Linear(fused_dim, 32), nn.ReLU(inplace=True), nn.Linear(32, 1))
    def forward(self, image: torch.Tensor, tabular: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        f_cnn = self.cnn_encoder(image)
        f_rad = self.rad_encoder(tabular)
        fused = self.fusion(f_cnn, f_rad)
        return self.clf_head(fused), self.reg_head(fused)


# Cell 4: Custom PyTorch dataset caching vectors to speed up training

In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import pandas as pd
class CachedHybridNoduleDataset(Dataset):
    def __init__(self, manifest_csv: str, exclude_indeterminate: bool = True):
        self.df = pd.read_csv(manifest_csv)
        if exclude_indeterminate:
            self.df = self.df[self.df["malignancy_class"] != -1].reset_index(drop=True)

        print(f"Caching 3D patches & extracting radiomics for {len(self.df)} nodules...")
        self.patches, self.tabular_features, self.labels, self.reg_scores, self.patient_ids = [], [], [], [], []
        for idx in tqdm(range(len(self.df)), desc="Extracting Features"):
            row = self.df.iloc[idx]
            data = torch.load(row["patch_path"], weights_only=False)
            patch_tensor = data["tensor"]
            patch_np = patch_tensor.squeeze(0).numpy()

            radiomics = extract_all_radiomics(patch_np)
            clinicals = np.array([
                row.get("subtlety", 0.0), row.get("sphericity", 0.0),
                row.get("margin", 0.0), row.get("spiculation", 0.0), row.get("texture", 0.0)
            ], dtype=np.float32)

            tabular_vector = np.concatenate([radiomics, clinicals])
            self.patches.append(patch_tensor)
            self.tabular_features.append(torch.tensor(tabular_vector, dtype=torch.float32))
            self.labels.append(int(row["malignancy_class"]))
            self.reg_scores.append(float(row["consensus_malignancy"]))
            self.patient_ids.append(row["patient_id"])
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int) -> dict:
        return {
            "image": self.patches[idx], "tabular": self.tabular_features[idx],
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "reg_score": torch.tensor(self.reg_scores[idx], dtype=torch.float32),
            "patient_id": self.patient_ids[idx]
        }


# Cell 5: GroupKFold cross-validation loops

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
import torch.optim as optim

def run_training_pipeline(epochs=15, batch_size=16, lr=3e-4, folds=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training running on: {device}")

    if device.type == "cpu":
        print("WARNING: Running on CPU. Go to Runtime > Change runtime type > Select T4 GPU in Colab for faster training!")

    full_dataset = CachedHybridNoduleDataset(str(MANIFEST_PATH))
    groups = full_dataset.patient_ids
    labels = np.array(full_dataset.labels)

    gkf = GroupKFold(n_splits=folds)

    for fold, (train_idx, val_idx) in enumerate(gkf.split(np.zeros(len(full_dataset)), labels, groups)):
        print(f"\n==================== Fold {fold+1}/{folds} ====================")

        # 1. Scale tabular features per fold
        scaler = StandardScaler()
        train_tab = np.array([full_dataset.tabular_features[i].numpy() for i in train_idx])
        val_tab = np.array([full_dataset.tabular_features[i].numpy() for i in val_idx])

        train_tab_scaled = scaler.fit_transform(train_tab)
        val_tab_scaled = scaler.transform(val_tab)

        for i, idx in enumerate(train_idx):
            full_dataset.tabular_features[idx] = torch.tensor(train_tab_scaled[i], dtype=torch.float32)
        for i, idx in enumerate(val_idx):
            full_dataset.tabular_features[idx] = torch.tensor(val_tab_scaled[i], dtype=torch.float32)

        # 2. Setup Loss with positive class weight balancing
        num_pos = np.sum(labels[train_idx] == 1)
        num_neg = np.sum(labels[train_idx] == 0)
        pos_weight = torch.tensor([1.0, num_neg / max(num_pos, 1)], dtype=torch.float32).to(device)

        clf_crit = nn.CrossEntropyLoss(weight=pos_weight)
        reg_crit = nn.MSELoss()

        train_sub = torch.utils.data.Subset(full_dataset, train_idx)
        val_sub = torch.utils.data.Subset(full_dataset, val_idx)

        train_loader = DataLoader(train_sub, batch_size=batch_size, shuffle=True, drop_last=False)
        val_loader = DataLoader(val_sub, batch_size=batch_size, shuffle=False)

        model = HybridMalignancyNet().to(device)
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

        # 3. Epoch Loop Execution
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for batch in train_loader:
                imgs = batch["image"].to(device)
                tabs = batch["tabular"].to(device)
                lbls = batch["label"].to(device)
                regs = batch["reg_score"].to(device)

                opt.zero_grad()
                clf_out, reg_out = model(imgs, tabs)

                loss_clf = clf_crit(clf_out, lbls)
                loss_reg = reg_crit(reg_out.squeeze(), regs)
                total_loss = loss_clf + 0.5 * loss_reg

                total_loss.backward()
                opt.step()
                train_loss += total_loss.item()

            print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {train_loss / len(train_loader):.4f}")

# Execute training
run_training_pipeline(epochs=15, batch_size=16)

Training running on: cuda
Caching 3D patches & extracting radiomics for 71 nodules...


Extracting Features:   0%|          | 0/71 [00:00<?, ?it/s]


==================== Fold 1/5 ====================
Epoch 01/15 | Train Loss: 5.3111
Epoch 02/15 | Train Loss: 5.4374
Epoch 03/15 | Train Loss: 4.9405
Epoch 04/15 | Train Loss: 4.7314
Epoch 05/15 | Train Loss: 4.0254
Epoch 06/15 | Train Loss: 3.4357
Epoch 07/15 | Train Loss: 2.3385
Epoch 08/15 | Train Loss: 1.7584
Epoch 09/15 | Train Loss: 1.0727
Epoch 10/15 | Train Loss: 1.4883
Epoch 11/15 | Train Loss: 1.1397
Epoch 12/15 | Train Loss: 1.1618
Epoch 13/15 | Train Loss: 0.9073
Epoch 14/15 | Train Loss: 0.7049
Epoch 15/15 | Train Loss: 0.9849

==================== Fold 2/5 ====================
Epoch 01/15 | Train Loss: 4.8981
Epoch 02/15 | Train Loss: 4.4126
Epoch 03/15 | Train Loss: 4.0799
Epoch 04/15 | Train Loss: 3.5515
Epoch 05/15 | Train Loss: 2.9766
Epoch 06/15 | Train Loss: 2.0462
Epoch 07/15 | Train Loss: 1.4351
Epoch 08/15 | Train Loss: 1.2350
Epoch 09/15 | Train Loss: 1.2810
Epoch 10/15 | Train Loss: 1.2279
Epoch 11/15 | Train Loss: 1.0426
Epoch 12/15 | Train Loss: 0.8960
Epoch